In [ ]:
# Import Models

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import scipy.stats as stats
import statsmodels.api as sm

In [3]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate the data as defined
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
error_Y = np.random.normal(0, 1, (1000,))
Y = X + Z + W + error_Y


In [4]:
# 1. Correlation with the structural error term
corr_structural = np.corrcoef(X, error_Y)[0, 1]

# 2. Correlation with the regression residual (OLS error term)
slope, intercept, r_value, p_value, std_err = stats.linregress(X, Y)
predicted_Y = intercept + slope * X
residuals = Y - predicted_Y
corr_residual = np.corrcoef(X, residuals)[0, 1]



In [5]:
print(f"Correlation with structural error: {corr_structural:.4f}")
print(f"Correlation with OLS regression residual: {corr_residual:.4f}")

Correlation with structural error: -0.0494
Correlation with OLS regression residual: -0.0000


In [6]:
# Set random seed for reproducibility
np.random.seed(42)
n_samples = 100000  # Large sample size to match expected population values

# Define the data generating process
W = np.random.normal(0, 1, (n_samples,))
X = W + np.random.normal(0, 1, (n_samples,)) 
Z = np.random.normal(0, 1, (n_samples,)) 
noise_Y = np.random.normal(0, 1, (n_samples,))

# The original structural equation for Y:
Y = X + Z + W + noise_Y

# If Y is written as depending on X and Z only, W becomes part of the error term:
# Y = X + Z + (W + noise_Y)
error_term = W + noise_Y

# Calculate the correlation between X and this error term
correlation = np.corrcoef(X, error_term)[0, 1]
print(f"Expected Correlation: {correlation:.2f}")

Expected Correlation: 0.50


In [ ]:
# 1. Recreate the underlying process data frame
np.random.seed(42)
n_samples = 100000

W = np.random.normal(0, 1, n_samples)
X = W + np.random.normal(0, 1, n_samples) 
Z = np.random.normal(0, 1, n_samples) 
Y = X + Z + W + np.random.normal(0, 1, n_samples)

df = pd.DataFrame({'W': W, 'X': X, 'Z': Z, 'Y': Y})

# 2. Define a function to fit regression in a narrow band around target W values
def get_x_coefficient(w_target, tolerance=0.1):
    # Filter data where W is within [w_target - tolerance, w_target + tolerance]
    subset = df[(df['W'] >= w_target - tolerance) & (df['W'] <= w_target + tolerance)]
    
    # Regress Y on X and Z
    features = sm.add_constant(subset[['X', 'Z']])
    model = sm.OLS(subset['Y'], features).fit()
    
    return model.params['X']

# Test the three target values of W
coef_neg1 = get_x_coefficient(-1.0)
coef_zero = get_x_coefficient(0.0)
coef_pos1 = get_x_coefficient(1.0)

print(f"Coefficient of X when W ≈ -1: {coef_neg1:.4f}")
print(f"Coefficient of X when W ≈  0: {coef_zero:.4f}")
print(f"Coefficient of X when W ≈  1: {coef_pos1:.4f}")

Coefficient of X when W ≈ -1: 1.0052
Coefficient of X when W ≈  0: 0.9884
Coefficient of X when W ≈  1: 1.0039


In [8]:
def make_error(corr_const, num): 
    err = list() 
    prev = np.random.normal(0, 1) 
    for n in range(num): 
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
        err.append(prev) 
    return np.array(err) 

# Simulation parameters
num_samples = 500
num_trials = 500
corr_values = [0.2, 0.5, 0.8]

for c in corr_values:
    coefs = []
    std_errors = []
    
    for _ in range(num_trials):
        # Generate autocorrelated errors for X and Y
        err_X = make_error(c, num_samples)
        err_Y = make_error(c, num_samples)
        
        # Treatment X and Outcome Y
        X = err_X
        Y = 1.0 * X + err_Y  # True beta = 1.0
        
        # Fit OLS with an intercept
        X_with_intercept = sm.add_constant(X)
        model = sm.OLS(Y, X_with_intercept).fit()
        
        coefs.append(model.params[1])
        std_errors.append(model.bse[1])
        
    # (i) True standard deviation of the coefficient across trials
    true_sd = np.std(coefs)
    # (ii) Mean of the estimated OLS standard errors
    mean_se = np.mean(std_errors)
    ratio = true_sd / mean_se
    
    print(f"corr_const = {c}:")
    print(f"  (i) True SD of estimate: {true_sd:.4f}")
    print(f"  (ii) Mean OLS Std Error: {mean_se:.4f}")
    print(f"  Ratio (i) / (ii): {ratio:.2f}\n")

corr_const = 0.2:
  (i) True SD of estimate: 0.0460
  (ii) Mean OLS Std Error: 0.0448
  Ratio (i) / (ii): 1.03

corr_const = 0.5:
  (i) True SD of estimate: 0.0575
  (ii) Mean OLS Std Error: 0.0449
  Ratio (i) / (ii): 1.28

corr_const = 0.8:
  (i) True SD of estimate: 0.0984
  (ii) Mean OLS Std Error: 0.0453
  Ratio (i) / (ii): 2.17

